In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
from sklearn.preprocessing import label_binarize

from catboost import CatBoostClassifier
from xgboost import XGBClassifier

import torch

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

HOURLY_PARQUET = "hourly_resampled_eicu/eicu_hourly_24h_label_idx_last_zero.parquet"
COHORT_PATH = "cohort/eicu_cohort_labeled.parquet"
SPLIT_PKL_DIR = Path("datasets/new_data_preparation")
WINDOW_HOURS = 24
FOLD = 0

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [2]:
def summarize_seeds(df, metrics, decimals=4):
    grouped = df.groupby("label")[metrics].agg(["mean", "std"])
    out = pd.DataFrame(index=grouped.index)
    for m in metrics:
        mean_s = grouped[(m, "mean")].round(decimals)
        std_s = grouped[(m, "std")].round(decimals)
        out[m] = mean_s.astype(str) + " ± " + std_s.astype(str)
    return out.reset_index()

In [3]:
hourly = pd.read_parquet(HOURLY_PARQUET)
cohort = pd.read_parquet(COHORT_PATH)

hourly.shape, cohort.shape

((3061176, 148), (132572, 74))

# Baseline Models

In [5]:
def load_split_stay_ids(path: Path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return [k for k in data.keys() if k != "_meta"]

train_ids = load_split_stay_ids(SPLIT_PKL_DIR / f"finetune_train_{WINDOW_HOURS}_fold{FOLD}_eicu.pkl")
val_ids   = load_split_stay_ids(SPLIT_PKL_DIR / f"finetune_val_{WINDOW_HOURS}_fold{FOLD}_eicu.pkl")
test_ids  = load_split_stay_ids(SPLIT_PKL_DIR / f"finetune_test_{WINDOW_HOURS}_eicu.pkl")

len(train_ids), len(val_ids), len(test_ids)

(81631, 20408, 25510)

In [7]:
def build_stay_level_features(hourly: pd.DataFrame) -> pd.DataFrame:
    value_cols = [c for c in hourly.columns if not c.startswith(("mask_", "delta_")) and c not in {"stay_id", "hour", "age", "gender"}]
    mask_cols = [c for c in hourly.columns if c.startswith("mask_")]

    # Tree 계열 모델 aggregation을 위한 변수 생성
    grouped = hourly.groupby("stay_id")
    value_stats = grouped[value_cols].agg(["last", "mean", "min", "max"])
    value_stats.columns = [f"{col}__{stat}" for col, stat in value_stats.columns]

    # 관측 여부를 관측 비율로 변환함.
    obs_rate = grouped[mask_cols].mean()
    obs_rate.columns = [f"{col}__rate" for col in obs_rate.columns]

    static = grouped[["age", "gender"]].first()

    features = pd.concat([static, value_stats, obs_rate], axis=1).fillna(0.0)
    return features.reset_index()

features = build_stay_level_features(hourly)
feature_cols = [c for c in features.columns if c != "stay_id"]

In [8]:
def subset(ids, merged, feature_cols, label):
    rows = merged[merged["stay_id"].isin(ids)]
    return rows[feature_cols], rows[label]

SEEDS = [41, 42, 43]

NUM_SOFA_CLASSES = 4

BINARY_LABELS = [
    "mortality_inicu", "mortality_48hr", "los_3days", "los_7days", "readmission_30",
    "transfusion_12hr", "shock_8hr", "vasopressor_need_12hr", "ventilator_need_12hr",
]

SOFA_LABELS = [
    "SOFA_centralnervous_24hr", "SOFA_cardiovascular_24hr", "SOFA_respiratory_24hr",
    "SOFA_coagulation_24hr", "SOFA_liver_24hr", "SOFA_renal_24hr",
]

PHENOTYPE_LABELS = [
    "Acute and unspecified renal failure", "Acute cerebrovascular disease",
    "Acute myocardial infarction", "Cardiac dysrhythmias", "Chronic kidney disease",
    "Chronic obstructive pulmonary disease and bronchiectasis",
    "Complications of surgical procedures or medical care", "Conduction disorders",
    "Congestive heart failure; nonhypertensive", "Coronary atherosclerosis and other heart disease",
    "Diabetes mellitus with complications", "Diabetes mellitus without complication",
    "Disorders of lipid metabolism", "Essential hypertension", "Fluid and electrolyte disorders",
    "Gastrointestinal hemorrhage", "Hypertension with complications and secondary hypertension",
    "Other liver diseases", "Other lower respiratory disease", "Other upper respiratory disease",
    "Pleurisy; pneumothorax; pulmonary collapse",
    "Pneumonia (except that caused by tuberculosis or sexually transmitted disease)",
    "Respiratory failure; insufficiency; arrest (adult)", "Septicemia (except in labor)", "Shock",
]

## 1. CatBoost

In [10]:
catboost_binary_results = []

for label in BINARY_LABELS:
    if label not in cohort.columns:
        continue
    merged = features.merge(cohort[["stay_id", label]], on="stay_id", how="inner").dropna(subset=[label])
    
    X_train, y_train = subset(train_ids, merged, feature_cols, label)
    X_val, y_val = subset(val_ids, merged, feature_cols, label)
    X_test, y_test = subset(test_ids, merged, feature_cols, label)
    
    if y_train.nunique() != 2 or X_train.empty or X_test.empty:
        print(f"[skip] {label}")
        continue

    for seed in SEEDS:
        model = CatBoostClassifier(
            iterations=1000, loss_function="Logloss", eval_metric="AUC",
            random_seed=seed, early_stopping_rounds=50, verbose=False,
        )
        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
        proba = model.predict_proba(X_test)[:, 1]
        catboost_binary_results.append({
            "label": label,
            "seed": seed,
            "n_train": len(y_train),
            "n_test": len(y_test),
            "auroc": roc_auc_score(y_test, proba),
            "auprc": average_precision_score(y_test, proba),
        })

catboost_binary_results = pd.DataFrame(catboost_binary_results)

summarize_seeds(catboost_binary_results, ["auroc", "auprc"])

,label,auroc,auprc
0,los_3days,0.7405 ± 0.0004,0.6366 ± 0.0013
1,los_7days,0.7795 ± 0.0012,0.3215 ± 0.0036
2,mortality_48hr,0.8753 ± 0.0018,0.2836 ± 0.0042
3,mortality_inicu,0.8613 ± 0.0004,0.3501 ± 0.007
4,readmission_30,0.6408 ± 0.0007,0.1547 ± 0.0019
5,shock_8hr,0.9365 ± 0.0041,0.2303 ± 0.002
6,transfusion_12hr,0.8534 ± 0.0009,0.0838 ± 0.0019
7,vasopressor_need_12hr,0.8862 ± 0.0006,0.4252 ± 0.006
8,ventilator_need_12hr,0.999 ± 0.0002,0.5344 ± 0.0311


In [11]:
catboost_multiclass_results = []

for label in SOFA_LABELS:
    if label not in cohort.columns:
        continue
    merged = features.merge(cohort[["stay_id", label]], on="stay_id", how="inner").dropna(subset=[label])

    X_train, y_train = subset(train_ids, merged, feature_cols, label)
    X_val, y_val = subset(val_ids, merged, feature_cols, label)
    X_test, y_test = subset(test_ids, merged, feature_cols, label)

    if y_train.nunique() < 2 or X_train.empty or X_test.empty:
        print(f"[skip] {label}")
        continue

    for seed in SEEDS:
        model = CatBoostClassifier(
            iterations=1000, loss_function="MultiClass", eval_metric="MultiClass",
            random_seed=seed, early_stopping_rounds=50, verbose=False,
        )
        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
        proba_full = model.predict_proba(X_test)  # (n_samples, model.classes_ 순서)

        y_test_arr = y_test.to_numpy()
        unique_labels = np.unique(y_test_arr)
        y_true_bin = label_binarize(y_test_arr, classes=np.arange(NUM_SOFA_CLASSES))
        y_true_subset = y_true_bin[:, unique_labels]

        class_to_col = {c: i for i, c in enumerate(model.classes_)}
        cols = [class_to_col[c] for c in unique_labels]
        prob_subset = proba_full[:, cols]

        catboost_multiclass_results.append({
            "label": label, "seed": seed,
            "n_train": len(y_train), "n_test": len(y_test),
            "auroc_macro": roc_auc_score(y_true_subset, prob_subset, average="macro"),
            # "auroc_micro": roc_auc_score(y_true_subset, prob_subset, average="micro"),
            "auprc_macro": average_precision_score(y_true_subset, prob_subset, average="macro"),
            # "auprc_micro": average_precision_score(y_true_subset, prob_subset, average="micro"),
        })

catboost_multiclass_results = pd.DataFrame(catboost_multiclass_results)
summarize_seeds(catboost_multiclass_results, ["auroc_macro", "auprc_macro"])
# summarize_seeds(catboost_multiclass_results, ["auroc_macro", "auroc_micro", "auprc_macro", "auprc_micro"])

,label,auroc_macro,auprc_macro
0,SOFA_cardiovascular_24hr,0.8874 ± 0.0014,0.4207 ± 0.0016
1,SOFA_centralnervous_24hr,0.7744 ± 0.0008,0.4231 ± 0.001
2,SOFA_coagulation_24hr,0.9037 ± 0.0002,0.5363 ± 0.0022
3,SOFA_liver_24hr,0.9184 ± 0.0008,0.4838 ± 0.0016
4,SOFA_renal_24hr,0.9158 ± 0.0002,0.5721 ± 0.002
5,SOFA_respiratory_24hr,0.867 ± 0.0008,0.525 ± 0.0015


In [12]:
catboost_phenotype_results = []
catboost_phenotype_raw = {}

for label in PHENOTYPE_LABELS:
    if label not in cohort.columns:
        continue
    merged = features.merge(cohort[["stay_id", label]], on="stay_id", how="inner").dropna(subset=[label])

    X_train, y_train = subset(train_ids, merged, feature_cols, label)
    X_val, y_val = subset(val_ids, merged, feature_cols, label)
    X_test, y_test = subset(test_ids, merged, feature_cols, label)

    if y_train.nunique() != 2 or X_train.empty or X_test.empty:
        print(f"[skip] {label}")
        continue

    for seed in SEEDS:
        model = CatBoostClassifier(
            iterations=1000, loss_function="Logloss", eval_metric="AUC",
            random_seed=seed, early_stopping_rounds=50, verbose=False,
        )
        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
        proba = model.predict_proba(X_test)[:, 1]
        catboost_phenotype_results.append({
            "label": label, "seed": seed,
            "n_train": len(y_train), "n_test": len(y_test),
            "auroc": roc_auc_score(y_test, proba),
            "auprc": average_precision_score(y_test, proba),
        })
        
        catboost_phenotype_raw[(label, seed)] = (y_test.to_numpy(), proba)

catboost_phenotype_results = pd.DataFrame(catboost_phenotype_results)

# 개별 phenotype별 계산
per_label_summary = summarize_seeds(catboost_phenotype_results, ["auroc", "auprc"])

macro_per_seed = catboost_phenotype_results.groupby("seed")[["auroc", "auprc"]].mean()
macro_mean, macro_std = macro_per_seed.mean(), macro_per_seed.std()

# micro
# included_labels = catboost_phenotype_results["label"].unique().tolist()
# micro_rows = []
# for seed in SEEDS:
#     y_all = np.concatenate([catboost_phenotype_raw[(label, seed)][0] for label in included_labels])
#     p_all = np.concatenate([catboost_phenotype_raw[(label, seed)][1] for label in included_labels])
#     micro_rows.append({
#         "seed": seed,
#         "auroc": roc_auc_score(y_all, p_all),
#         "auprc": average_precision_score(y_all, p_all),
#     })
# micro_per_seed = pd.DataFrame(micro_rows).set_index("seed")
# micro_mean, micro_std = micro_per_seed.mean(), micro_per_seed.std()

overall_rows = pd.DataFrame([
    {
        "label": "OVERALL (macro avg)",
        "auroc": f"{macro_mean['auroc']:.4f} ± {macro_std['auroc']:.4f}",
        "auprc": f"{macro_mean['auprc']:.4f} ± {macro_std['auprc']:.4f}",
    },
    # {
    #     "label": "OVERALL (micro avg)",
    #     "auroc": f"{micro_mean['auroc']:.4f} ± {micro_std['auroc']:.4f}",
    #     "auprc": f"{micro_mean['auprc']:.4f} ± {micro_std['auprc']:.4f}",
    # },
])

phenotype_summary = pd.concat([per_label_summary, overall_rows], ignore_index=True)
phenotype_summary

,label,auroc,auprc
0,Acute and unspecified renal failure,0.863 ± 0.0006,0.4976 ± 0.0019
1,Acute cerebrovascular disease,0.8375 ± 0.0011,0.3845 ± 0.0058
2,Acute myocardial infarction,0.8362 ± 0.0015,0.3839 ± 0.0002
3,Cardiac dysrhythmias,0.7647 ± 0.0018,0.3734 ± 0.0026
4,Chronic kidney disease,0.8966 ± 0.0004,0.5134 ± 0.002
5,Chronic obstructive pulmonary disease and bron...,0.8122 ± 0.0044,0.3443 ± 0.0053
6,Complications of surgical procedures or medica...,0.7234 ± 0.0049,0.0256 ± 0.0021
7,Conduction disorders,0.7646 ± 0.0038,0.0807 ± 0.002
8,Congestive heart failure; nonhypertensive,0.8078 ± 0.0016,0.3438 ± 0.0032
9,Coronary atherosclerosis and other heart disease,0.7993 ± 0.0044,0.1417 ± 0.0029


## 2. XGBoost

In [7]:
xgboost_binary_results = []

for label in BINARY_LABELS:
    if label not in cohort.columns:
        continue
    merged = features.merge(cohort[["stay_id", label]], on="stay_id", how="inner").dropna(subset=[label])
    
    X_train, y_train = subset(train_ids, merged, feature_cols, label)
    X_val, y_val = subset(val_ids, merged, feature_cols, label)
    X_test, y_test = subset(test_ids, merged, feature_cols, label)
    
    if y_train.nunique() != 2 or X_train.empty or X_test.empty:
        print(f"[skip] {label}")
        continue


    for seed in SEEDS:
        model = XGBClassifier(
            n_estimators=1000, objective="binary:logistic", eval_metric="auc",
            random_state=seed, early_stopping_rounds=50, verbosity=0,
            subsample=0.8, colsample_bytree=0.8,
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        proba = model.predict_proba(X_test)[:, 1]
        xgboost_binary_results.append({
            "label": label,
            "seed": seed,
            "n_train": len(y_train),
            "n_test": len(y_test),
            "auroc": roc_auc_score(y_test, proba),
            "auprc": average_precision_score(y_test, proba),
        })

xgboost_binary_results = pd.DataFrame(xgboost_binary_results)

summarize_seeds(xgboost_binary_results, ["auroc", "auprc"])

,label,auroc,auprc
0,los_3days,0.732 ± 0.0007,0.6242 ± 0.0033
1,los_7days,0.7694 ± 0.0008,0.2991 ± 0.002
2,mortality_48hr,0.8618 ± 0.005,0.2404 ± 0.0153
3,mortality_inicu,0.8521 ± 0.0029,0.312 ± 0.0074
4,readmission_30,0.6226 ± 0.0021,0.1421 ± 0.003
5,shock_8hr,0.9242 ± 0.0066,0.1876 ± 0.0057
6,transfusion_12hr,0.8421 ± 0.0033,0.0759 ± 0.0008
7,vasopressor_need_12hr,0.8802 ± 0.0009,0.3926 ± 0.0005
8,ventilator_need_12hr,0.9976 ± 0.0006,0.5038 ± 0.0339


In [8]:
xgboost_multiclass_results = []

NUM_SOFA_CLASSES = 4

for label in SOFA_LABELS:
    if label not in cohort.columns:
        continue
    merged = features.merge(cohort[["stay_id", label]], on="stay_id", how="inner").dropna(subset=[label])

    X_train, y_train = subset(train_ids, merged, feature_cols, label)
    X_val, y_val = subset(val_ids, merged, feature_cols, label)
    X_test, y_test = subset(test_ids, merged, feature_cols, label)

    if y_train.nunique() < 2 or X_train.empty or X_test.empty:
        print(f"[skip] {label}")
        continue

    for seed in SEEDS:
        model = XGBClassifier(
            n_estimators=1000, objective="multi:softprob", eval_metric="mlogloss",
            random_state=seed, early_stopping_rounds=50, verbosity=0,
            subsample=0.8, colsample_bytree=0.8,
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        proba_full = model.predict_proba(X_test)

        y_test_arr = y_test.to_numpy()
        unique_labels = np.unique(y_test_arr)
        y_true_bin = label_binarize(y_test_arr, classes=np.arange(NUM_SOFA_CLASSES))
        y_true_subset = y_true_bin[:, unique_labels]

        class_to_col = {c: i for i, c in enumerate(model.classes_)}
        cols = [class_to_col[c] for c in unique_labels]
        prob_subset = proba_full[:, cols]

        xgboost_multiclass_results.append({
            "label": label, "seed": seed,
            "n_train": len(y_train), "n_test": len(y_test),
            "auroc_macro": roc_auc_score(y_true_subset, prob_subset, average="macro"),
            # "auroc_micro": roc_auc_score(y_true_subset, prob_subset, average="micro"),
            "auprc_macro": average_precision_score(y_true_subset, prob_subset, average="macro"),
            # "auprc_micro": average_precision_score(y_true_subset, prob_subset, average="micro"),
        })

xgboost_multiclass_results = pd.DataFrame(xgboost_multiclass_results)
summarize_seeds(xgboost_multiclass_results, ["auroc_macro", "auprc_macro"])
# summarize_seeds(xgboost_multiclass_results, ["auroc_macro", "auroc_micro", "auprc_macro", "auprc_micro"])

,label,auroc_macro,auprc_macro
0,SOFA_cardiovascular_24hr,0.8881 ± 0.0003,0.4059 ± 0.0042
1,SOFA_centralnervous_24hr,0.7685 ± 0.0005,0.4161 ± 0.0018
2,SOFA_coagulation_24hr,0.9007 ± 0.0002,0.5228 ± 0.0025
3,SOFA_liver_24hr,0.9177 ± 0.0016,0.4703 ± 0.0034
4,SOFA_renal_24hr,0.9136 ± 0.0006,0.5632 ± 0.0054
5,SOFA_respiratory_24hr,0.8651 ± 0.0017,0.5212 ± 0.0017


In [9]:
xgboost_phenotype_results = []
xgboost_phenotype_raw = {}

for label in PHENOTYPE_LABELS:
    if label not in cohort.columns:
        continue
    merged = features.merge(cohort[["stay_id", label]], on="stay_id", how="inner").dropna(subset=[label])

    X_train, y_train = subset(train_ids, merged, feature_cols, label)
    X_val, y_val = subset(val_ids, merged, feature_cols, label)
    X_test, y_test = subset(test_ids, merged, feature_cols, label)

    if y_train.nunique() != 2 or X_train.empty or X_test.empty:
        print(f"[skip] {label}")
        continue

    for seed in SEEDS:
        model = XGBClassifier(
            n_estimators=1000, objective="binary:logistic", eval_metric="auc",
            random_state=seed, early_stopping_rounds=50, verbosity=0,
            subsample=0.8, colsample_bytree=0.8,
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        
        proba = model.predict_proba(X_test)[:, 1]
        xgboost_phenotype_results.append({
            "label": label, "seed": seed,
            "n_train": len(y_train), "n_test": len(y_test),
            "auroc": roc_auc_score(y_test, proba),
            "auprc": average_precision_score(y_test, proba),
        })

        xgboost_phenotype_raw[(label, seed)] = (y_test.to_numpy(), proba)

xgboost_phenotype_results = pd.DataFrame(xgboost_phenotype_results)

# 개별 phenotype별 계산
xgboost_per_label_summary = summarize_seeds(xgboost_phenotype_results, ["auroc", "auprc"])

macro_per_seed = xgboost_phenotype_results.groupby("seed")[["auroc", "auprc"]].mean()
macro_mean, macro_std = macro_per_seed.mean(), macro_per_seed.std()

# micro
# included_labels = xgboost_phenotype_results["label"].unique().tolist()
# micro_rows = []
# for seed in SEEDS:
#     y_all = np.concatenate([xgboost_phenotype_raw[(label, seed)][0] for label in included_labels])
#     p_all = np.concatenate([xgboost_phenotype_raw[(label, seed)][1] for label in included_labels])
#     micro_rows.append({
#         "seed": seed,
#         "auroc": roc_auc_score(y_all, p_all),
#         "auprc": average_precision_score(y_all, p_all),
#     })
# micro_per_seed = pd.DataFrame(micro_rows).set_index("seed")
# micro_mean, micro_std = micro_per_seed.mean(), micro_per_seed.std()

overall_rows = pd.DataFrame([
    {
        "label": "OVERALL (macro avg)",
        "auroc": f"{macro_mean['auroc']:.4f} ± {macro_std['auroc']:.4f}",
        "auprc": f"{macro_mean['auprc']:.4f} ± {macro_std['auprc']:.4f}",
    },
    # {
    #     "label": "OVERALL (micro avg)",
    #     "auroc": f"{micro_mean['auroc']:.4f} ± {micro_std['auroc']:.4f}",
    #     "auprc": f"{micro_mean['auprc']:.4f} ± {micro_std['auprc']:.4f}",
    # },
])

xgboost_phenotype_summary = pd.concat([xgboost_per_label_summary, overall_rows], ignore_index=True)
xgboost_phenotype_summary

,label,auroc,auprc
0,Acute and unspecified renal failure,0.8534 ± 0.0015,0.464 ± 0.0043
1,Acute cerebrovascular disease,0.8238 ± 0.0024,0.3541 ± 0.005
2,Acute myocardial infarction,0.8207 ± 0.0013,0.3535 ± 0.0008
3,Cardiac dysrhythmias,0.7469 ± 0.0019,0.3461 ± 0.0019
4,Chronic kidney disease,0.8887 ± 0.0015,0.4901 ± 0.0049
5,Chronic obstructive pulmonary disease and bron...,0.8011 ± 0.0025,0.323 ± 0.0025
6,Complications of surgical procedures or medica...,0.693 ± 0.008,0.0204 ± 0.0011
7,Conduction disorders,0.7497 ± 0.0016,0.068 ± 0.004
8,Congestive heart failure; nonhypertensive,0.7917 ± 0.0026,0.3168 ± 0.005
9,Coronary atherosclerosis and other heart disease,0.7822 ± 0.0082,0.1284 ± 0.002


## 3. LSTM

In [9]:
WINDOW_HOURS = 24

ALL_BINARY = BINARY_LABELS
ALL_SOFA = SOFA_LABELS
ALL_PHENO = PHENOTYPE_LABELS

value_cols = [c for c in hourly.columns if not c.startswith(("mask_", "delta_")) and c not in {"stay_id", "hour", "age", "gender"}]
mask_cols = [f"mask_{c}" for c in value_cols]

# def build_sequence_tensors(hourly: pd.DataFrame, stay_ids):
#     hourly = hourly[hourly["stay_id"].isin(stay_ids)]
#     stay_order = [s for s in stay_ids if s in set(hourly["stay_id"])]
#     n = len(stay_order)
#     idx_map = {s: i for i, s in enumerate(stay_order)}

#     X = np.zeros((n, WINDOW_HOURS, len(value_cols)), dtype="float32")
#     M = np.zeros((n, WINDOW_HOURS, len(value_cols)), dtype="float32")

#     row_idx = hourly["stay_id"].map(idx_map).to_numpy()
#     hour_idx = hourly["hour"].to_numpy().astype(int)

#     X[row_idx, hour_idx] = hourly[value_cols].to_numpy(dtype="float32")
#     M[row_idx, hour_idx] = hourly[mask_cols].to_numpy(dtype="float32")

#     return stay_order, X, M

def build_sequence_tensors(hourly: pd.DataFrame, stay_ids):
    hourly = hourly[hourly["stay_id"].isin(stay_ids)]
    present = set(hourly["stay_id"].unique())
    stay_order = [s for s in stay_ids if s in present]
    n = len(stay_order)
    idx_map = {s: i for i, s in enumerate(stay_order)}

    X = np.zeros((n, WINDOW_HOURS, len(value_cols)), dtype="float32")
    M = np.zeros((n, WINDOW_HOURS, len(value_cols)), dtype="float32")

    row_idx = hourly["stay_id"].map(idx_map).to_numpy()
    hour_idx = hourly["hour"].to_numpy().astype(int)

    X[row_idx, hour_idx] = hourly[value_cols].to_numpy(dtype="float32")
    M[row_idx, hour_idx] = hourly[mask_cols].to_numpy(dtype="float32")

    return stay_order, X, M


In [10]:
n_features = len(value_cols) * 2

train_order, X_train_raw, M_train = build_sequence_tensors(hourly, train_ids)
val_order,   X_val_raw,   M_val   = build_sequence_tensors(hourly, val_ids)
test_order,  X_test_raw,  M_test  = build_sequence_tensors(hourly, test_ids)

train_flat = X_train_raw.reshape(-1, X_train_raw.shape[-1])
train_mask_flat = M_train.reshape(-1, M_train.shape[-1])
mean = np.array([train_flat[train_mask_flat[:, j] == 1, j].mean() if (train_mask_flat[:, j] == 1).any() else 0.0 for j in range(train_flat.shape[1])], dtype="float32")
std = np.array([train_flat[train_mask_flat[:, j] == 1, j].std() if (train_mask_flat[:, j] == 1).any() else 1.0 for j in range(train_flat.shape[1])], dtype="float32")
std[std == 0] = 1.0

def normalize(X, M):
    X_norm = (X - mean) / std
    X_norm[M == 0] = 0.0
    return X_norm

X_train = normalize(X_train_raw, M_train)
X_val = normalize(X_val_raw, M_val)
X_test = normalize(X_test_raw, M_test)

label_lookup_mt = cohort.set_index("stay_id")[ALL_BINARY + ALL_SOFA + ALL_PHENO]

class EicuMultiTaskDataset(Dataset):
    def __init__(self, stay_order, X, M, label_lookup):
        keep = [i for i, s in enumerate(stay_order) if s in label_lookup.index]
        self.X = X[keep]
        self.M = M[keep]
        rows = label_lookup.loc[[stay_order[i] for i in keep]]
        self.y_binary = rows[ALL_BINARY].to_numpy(dtype="float32")
        self.y_sofa = rows[ALL_SOFA].to_numpy(dtype="int64")
        self.y_pheno = rows[ALL_PHENO].to_numpy(dtype="float32")

    def __len__(self):
        return len(self.y_binary)

    def __getitem__(self, idx):
        x = np.concatenate([self.X[idx], self.M[idx]], axis=-1)
        return (
            torch.from_numpy(x),
            torch.from_numpy(self.y_binary[idx]),
            torch.from_numpy(self.y_sofa[idx]),
            torch.from_numpy(self.y_pheno[idx]),
        )

mt_train_ds = EicuMultiTaskDataset(train_order, X_train, M_train, label_lookup_mt)
mt_val_ds = EicuMultiTaskDataset(val_order, X_val, M_val, label_lookup_mt)
mt_test_ds = EicuMultiTaskDataset(test_order, X_test, M_test, label_lookup_mt)

mt_train_loader = DataLoader(mt_train_ds, batch_size=64, shuffle=True)
mt_val_loader = DataLoader(mt_val_ds, batch_size=256, shuffle=False)
mt_test_loader = DataLoader(mt_test_ds, batch_size=256, shuffle=False)

len(mt_train_ds), len(mt_val_ds), len(mt_test_ds)

(81631, 20408, 25510)

In [11]:
def multitask_loss(binary_logits, sofa_logits, pheno_logits, y_binary, y_sofa, y_pheno):
    binary_loss = F.binary_cross_entropy_with_logits(binary_logits, y_binary, reduction="none").sum(dim=1).mean()
    sofa_loss = sum(F.cross_entropy(sofa_logits[:, i, :], y_sofa[:, i]) for i in range(sofa_logits.shape[1]))
    pheno_loss = F.binary_cross_entropy_with_logits(pheno_logits, y_pheno)  # 기본 mean: 25개+batch 평균
    return binary_loss + sofa_loss + pheno_loss

def train_multitask(model, train_loader, val_loader, device, epochs=60, lr=2e-4, weight_decay=1e-5, patience=8):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_score, best_state, patience_left = -1, None, patience
    epoch_bar = tqdm(range(epochs), desc="epoch")
    for epoch in epoch_bar:
        model.train()
        batch_bar = tqdm(train_loader, desc="train", leave=False)
        for xb, yb_bin, yb_sofa, yb_pheno in batch_bar:
            xb, yb_bin, yb_sofa, yb_pheno = xb.to(device), yb_bin.to(device), yb_sofa.to(device), yb_pheno.to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                binary_logits, sofa_logits, pheno_logits = model(xb)
                loss = multitask_loss(binary_logits, sofa_logits, pheno_logits, yb_bin, yb_sofa, yb_pheno)
            loss.backward()
            optimizer.step()
            batch_bar.set_postfix(loss=loss.item())

        model.eval()
        val_probs, val_true = [], []
        with torch.no_grad():
            for xb, yb_bin, yb_sofa, yb_pheno in tqdm(val_loader, desc="val", leave=False):
                binary_logits, _, _ = model(xb.to(device))
                val_probs.append(torch.sigmoid(binary_logits).cpu().numpy())
                val_true.append(yb_bin.numpy())
        val_probs, val_true = np.concatenate(val_probs), np.concatenate(val_true)

        task_aurocs = [
            roc_auc_score(val_true[:, i], val_probs[:, i])
            for i in range(val_true.shape[1]) if len(np.unique(val_true[:, i])) == 2
        ]
        val_score = np.mean(task_aurocs) if task_aurocs else 0.0
        epoch_bar.set_postfix(val_auroc=f"{val_score:.4f}")

        if val_score > best_val_score:
            best_val_score, best_state, patience_left = val_score, {k: v.clone() for k, v in model.state_dict().items()}, patience
        else:
            patience_left -= 1
            if patience_left == 0:
                print("early stopping")
                break

    model.load_state_dict(best_state)
    return model, best_val_score

def evaluate_multitask(model, loader, device):
    model.eval()
    binary_probs, binary_true = [], []
    sofa_probs, sofa_true = [], []
    pheno_probs, pheno_true = [], []
    with torch.no_grad():
        for xb, yb_bin, yb_sofa, yb_pheno in tqdm(loader, desc="evaluate", leave=False):
            binary_logits, sofa_logits, pheno_logits = model(xb.to(device))
            binary_probs.append(torch.sigmoid(binary_logits).cpu().numpy())
            binary_true.append(yb_bin.numpy())
            sofa_probs.append(torch.softmax(sofa_logits, dim=-1).cpu().numpy())
            sofa_true.append(yb_sofa.numpy())
            pheno_probs.append(torch.sigmoid(pheno_logits).cpu().numpy())
            pheno_true.append(yb_pheno.numpy())

    binary_probs, binary_true = np.concatenate(binary_probs), np.concatenate(binary_true)
    sofa_probs, sofa_true = np.concatenate(sofa_probs), np.concatenate(sofa_true)
    pheno_probs, pheno_true = np.concatenate(pheno_probs), np.concatenate(pheno_true)

    binary_rows, sofa_rows, pheno_rows = [], [], []

    for i, label in enumerate(ALL_BINARY):
        binary_rows.append({
            "label": label,
            "auroc": roc_auc_score(binary_true[:, i], binary_probs[:, i]),
            "auprc": average_precision_score(binary_true[:, i], binary_probs[:, i]),
        })

    for i, label in enumerate(ALL_SOFA):
        y_true, proba = sofa_true[:, i], sofa_probs[:, i, :]
        unique_labels = np.unique(y_true)
        y_true_bin = label_binarize(y_true, classes=np.arange(NUM_SOFA_CLASSES))
        y_true_subset = y_true_bin[:, unique_labels]
        prob_subset = proba[:, unique_labels]
        sofa_rows.append({
            "label": label,
            "auroc_macro": roc_auc_score(y_true_subset, prob_subset, average="macro"),
            "auprc_macro": average_precision_score(y_true_subset, prob_subset, average="macro"),
        })

    for i, label in enumerate(ALL_PHENO):
        pheno_rows.append({
            "label": label,
            "auroc": roc_auc_score(pheno_true[:, i], pheno_probs[:, i]),
            "auprc": average_precision_score(pheno_true[:, i], pheno_probs[:, i]),
        })

    return binary_rows, sofa_rows, pheno_rows, pheno_true, pheno_probs

In [34]:
class MultiTaskLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2, n_binary=9, n_sofa=6, n_sofa_classes=4, n_pheno=25):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        
        self.binary_head = nn.Linear(hidden_size, n_binary)
        self.sofa_head = nn.Linear(hidden_size, n_sofa * n_sofa_classes)
        self.pheno_head = nn.Linear(hidden_size, n_pheno)
        
        self.n_sofa = n_sofa
        self.n_sofa_classes = n_sofa_classes

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h = self.dropout(h_n[-1])
        binary_logits = self.binary_head(h)                                          # (batch, 9)
        sofa_logits = self.sofa_head(h).view(-1, self.n_sofa, self.n_sofa_classes)   # (batch, 6, 4)
        pheno_logits = self.pheno_head(h)                                            # (batch, 25)
        return binary_logits, sofa_logits, pheno_logits

In [36]:
lstm_binary_results, lstm_sofa_results, lstm_phenotype_results = [], [], []
lstm_phenotype_raw = {}

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MultiTaskLSTM(input_size=n_features)
    model, best_val_score = train_multitask(model, mt_train_loader, mt_val_loader, device)
    binary_rows, sofa_rows, pheno_rows, pheno_true, pheno_probs = evaluate_multitask(model, mt_test_loader, device)

    for row in binary_rows:
        lstm_binary_results.append({**row, "seed": seed})
    for row in sofa_rows:
        lstm_sofa_results.append({**row, "seed": seed})
    for i, label in enumerate(ALL_PHENO):
        lstm_phenotype_results.append({**pheno_rows[i], "seed": seed})
        lstm_phenotype_raw[(label, seed)] = (pheno_true[:, i], pheno_probs[:, i])

lstm_binary_results = pd.DataFrame(lstm_binary_results)
lstm_sofa_results = pd.DataFrame(lstm_sofa_results)
lstm_phenotype_results = pd.DataFrame(lstm_phenotype_results)

summarize_seeds(lstm_binary_results, ["auroc", "auprc"])

epoch:   0%|          | 0/50 [00:00<?, ?it/s]

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

...

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

early stopping


evaluate:   0%|          | 0/100 [00:00<?, ?it/s]

,label,auroc,auprc
0,los_3days,0.7509 ± 0.0001,0.6564 ± 0.0003
1,los_7days,0.7909 ± 0.0006,0.3325 ± 0.0035
2,mortality_48hr,0.8828 ± 0.0019,0.3086 ± 0.0003
3,mortality_inicu,0.8632 ± 0.0028,0.3548 ± 0.0021
4,readmission_30,0.624 ± 0.0012,0.1251 ± 0.0018
5,shock_8hr,0.9251 ± 0.0015,0.1789 ± 0.0045
6,transfusion_12hr,0.8211 ± 0.0046,0.0715 ± 0.0041
7,vasopressor_need_12hr,0.8872 ± 0.0033,0.4107 ± 0.0127
8,ventilator_need_12hr,0.9734 ± 0.0076,0.1241 ± 0.024


In [37]:
summarize_seeds(lstm_sofa_results, ["auroc_macro", "auprc_macro"])

,label,auroc_macro,auprc_macro
0,SOFA_cardiovascular_24hr,0.8827 ± 0.0011,0.3697 ± 0.007
1,SOFA_centralnervous_24hr,0.7535 ± 0.0008,0.4056 ± 0.0017
2,SOFA_coagulation_24hr,0.9036 ± 0.0006,0.5341 ± 0.0037
3,SOFA_liver_24hr,0.9103 ± 0.0004,0.4565 ± 0.0028
4,SOFA_renal_24hr,0.9161 ± 0.0016,0.5826 ± 0.0048
5,SOFA_respiratory_24hr,0.8638 ± 0.0012,0.5216 ± 0.0011


In [38]:
lstm_per_label_summary = summarize_seeds(lstm_phenotype_results, ["auroc", "auprc"])

macro_per_seed = lstm_phenotype_results.groupby("seed")[["auroc", "auprc"]].mean()
macro_mean, macro_std = macro_per_seed.mean(), macro_per_seed.std()

# included_labels = lstm_phenotype_results["label"].unique().tolist()
# micro_rows = []
# for seed in SEEDS:
#     y_all = np.concatenate([lstm_phenotype_raw[(label, seed)][0] for label in included_labels])
#     p_all = np.concatenate([lstm_phenotype_raw[(label, seed)][1] for label in included_labels])
#     micro_rows.append({
#         "seed": seed,
#         "auroc": roc_auc_score(y_all, p_all),
#         "auprc": average_precision_score(y_all, p_all),
#     })
# micro_per_seed = pd.DataFrame(micro_rows).set_index("seed")
# micro_mean, micro_std = micro_per_seed.mean(), micro_per_seed.std()

overall_rows = pd.DataFrame([
    {
        "label": "OVERALL (macro avg)",
        "auroc": f"{macro_mean['auroc']:.4f} ± {macro_std['auroc']:.4f}",
        "auprc": f"{macro_mean['auprc']:.4f} ± {macro_std['auprc']:.4f}",
    },
    # {
    #     "label": "OVERALL (micro avg)",
    #     "auroc": f"{micro_mean['auroc']:.4f} ± {micro_std['auroc']:.4f}",
    #     "auprc": f"{micro_mean['auprc']:.4f} ± {micro_std['auprc']:.4f}",
    # },
])

lstm_phenotype_summary = pd.concat([lstm_per_label_summary, overall_rows], ignore_index=True)
lstm_phenotype_summary

,label,auroc,auprc
0,Acute and unspecified renal failure,0.8165 ± 0.0021,0.3671 ± 0.0052
1,Acute cerebrovascular disease,0.7688 ± 0.0019,0.2317 ± 0.0033
2,Acute myocardial infarction,0.6943 ± 0.0061,0.13 ± 0.0058
3,Cardiac dysrhythmias,0.6367 ± 0.0036,0.2113 ± 0.004
4,Chronic kidney disease,0.8326 ± 0.0006,0.3527 ± 0.003
5,Chronic obstructive pulmonary disease and bron...,0.7337 ± 0.009,0.2292 ± 0.013
6,Complications of surgical procedures or medica...,0.6673 ± 0.0097,0.0171 ± 0.0006
7,Conduction disorders,0.6893 ± 0.0123,0.0257 ± 0.003
8,Congestive heart failure; nonhypertensive,0.7389 ± 0.004,0.2382 ± 0.0052
9,Coronary atherosclerosis and other heart disease,0.7008 ± 0.0047,0.0645 ± 0.0048


## 4. TSMixer

In [39]:
from tsmixer.tsmixer import TSMixer

class MultiTaskTSMixer(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_blocks=2, dropout=0.2, n_binary=9, n_sofa=6, n_sofa_classes=4, n_pheno=25):
        super().__init__()
        self.backbone = TSMixer(
            sequence_length=WINDOW_HOURS,
            prediction_length=1,
            input_channels=input_size,
            output_channels=hidden_size,
            num_blocks=num_blocks,
            dropout_rate=dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.binary_head = nn.Linear(hidden_size, n_binary)
        self.sofa_head = nn.Linear(hidden_size, n_sofa * n_sofa_classes)
        self.pheno_head = nn.Linear(hidden_size, n_pheno)
        self.n_sofa = n_sofa
        self.n_sofa_classes = n_sofa_classes

    def forward(self, x):
        h = self.backbone(x).squeeze(1)
        h = self.dropout(h)
        binary_logits = self.binary_head(h)
        sofa_logits = self.sofa_head(h).view(-1, self.n_sofa, self.n_sofa_classes)
        pheno_logits = self.pheno_head(h)
        return binary_logits, sofa_logits, pheno_logits

In [40]:
tsmixer_binary_results, tsmixer_sofa_results, tsmixer_phenotype_results = [], [], []
tsmixer_phenotype_raw = {}

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MultiTaskTSMixer(input_size=n_features)
    model, best_val_score = train_multitask(model, mt_train_loader, mt_val_loader, device)
    binary_rows, sofa_rows, pheno_rows, pheno_true, pheno_probs = evaluate_multitask(model, mt_test_loader, device)

    for row in binary_rows:
        tsmixer_binary_results.append({**row, "seed": seed})
    for row in sofa_rows:
        tsmixer_sofa_results.append({**row, "seed": seed})
    for i, label in enumerate(ALL_PHENO):
        tsmixer_phenotype_results.append({**pheno_rows[i], "seed": seed})
        tsmixer_phenotype_raw[(label, seed)] = (pheno_true[:, i], pheno_probs[:, i])

tsmixer_binary_results = pd.DataFrame(tsmixer_binary_results)
tsmixer_sofa_results = pd.DataFrame(tsmixer_sofa_results)
tsmixer_phenotype_results = pd.DataFrame(tsmixer_phenotype_results)

summarize_seeds(tsmixer_binary_results, ["auroc", "auprc"])

epoch:   0%|          | 0/50 [00:00<?, ?it/s]

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

...

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

early stopping


evaluate:   0%|          | 0/100 [00:00<?, ?it/s]

,label,auroc,auprc
0,los_3days,0.737 ± 0.0014,0.6353 ± 0.0042
1,los_7days,0.7797 ± 0.0019,0.3165 ± 0.0061
2,mortality_48hr,0.8688 ± 0.0041,0.2187 ± 0.0159
3,mortality_inicu,0.8556 ± 0.0021,0.3106 ± 0.0179
4,readmission_30,0.6312 ± 0.0031,0.1295 ± 0.0036
5,shock_8hr,0.9334 ± 0.0014,0.2298 ± 0.0117
6,transfusion_12hr,0.8418 ± 0.0065,0.0876 ± 0.0029
7,vasopressor_need_12hr,0.8768 ± 0.0023,0.381 ± 0.0107
8,ventilator_need_12hr,0.9917 ± 0.0013,0.2195 ± 0.0502


In [41]:
summarize_seeds(tsmixer_sofa_results, ["auroc_macro", "auprc_macro"])

,label,auroc_macro,auprc_macro
0,SOFA_cardiovascular_24hr,0.8775 ± 0.0027,0.3833 ± 0.0071
1,SOFA_centralnervous_24hr,0.7432 ± 0.0024,0.3945 ± 0.0038
2,SOFA_coagulation_24hr,0.8832 ± 0.0029,0.469 ± 0.0098
3,SOFA_liver_24hr,0.9036 ± 0.0049,0.4515 ± 0.0114
4,SOFA_renal_24hr,0.8979 ± 0.0013,0.5254 ± 0.0037
5,SOFA_respiratory_24hr,0.8638 ± 0.0012,0.5218 ± 0.0022


In [42]:
tsmixer_per_label_summary = summarize_seeds(tsmixer_phenotype_results, ["auroc", "auprc"])

macro_per_seed = tsmixer_phenotype_results.groupby("seed")[["auroc", "auprc"]].mean()
macro_mean, macro_std = macro_per_seed.mean(), macro_per_seed.std()

# included_labels = tsmixer_phenotype_results["label"].unique().tolist()
# micro_rows = []
# for seed in SEEDS:
#     y_all = np.concatenate([lstm_phenotype_raw[(label, seed)][0] for label in included_labels])
#     p_all = np.concatenate([lstm_phenotype_raw[(label, seed)][1] for label in included_labels])
#     micro_rows.append({
#         "seed": seed,
#         "auroc": roc_auc_score(y_all, p_all),
#         "auprc": average_precision_score(y_all, p_all),
#     })
# micro_per_seed = pd.DataFrame(micro_rows).set_index("seed")
# micro_mean, micro_std = micro_per_seed.mean(), micro_per_seed.std()

overall_rows = pd.DataFrame([
    {
        "label": "OVERALL (macro avg)",
        "auroc": f"{macro_mean['auroc']:.4f} ± {macro_std['auroc']:.4f}",
        "auprc": f"{macro_mean['auprc']:.4f} ± {macro_std['auprc']:.4f}",
    },
    # {
    #     "label": "OVERALL (micro avg)",
    #     "auroc": f"{micro_mean['auroc']:.4f} ± {micro_std['auroc']:.4f}",
    #     "auprc": f"{micro_mean['auprc']:.4f} ± {micro_std['auprc']:.4f}",
    # },
])

tsmixer_phenotype_summary = pd.concat([tsmixer_per_label_summary, overall_rows], ignore_index=True)
tsmixer_phenotype_summary

,label,auroc,auprc
0,Acute and unspecified renal failure,0.8296 ± 0.0005,0.3955 ± 0.0025
1,Acute cerebrovascular disease,0.7885 ± 0.0025,0.2484 ± 0.0038
2,Acute myocardial infarction,0.7592 ± 0.0035,0.246 ± 0.0041
3,Cardiac dysrhythmias,0.6593 ± 0.0039,0.2375 ± 0.0042
4,Chronic kidney disease,0.8385 ± 0.001,0.3704 ± 0.0038
5,Chronic obstructive pulmonary disease and bron...,0.7602 ± 0.0033,0.2716 ± 0.0093
6,Complications of surgical procedures or medica...,0.7184 ± 0.0147,0.0249 ± 0.002
7,Conduction disorders,0.7122 ± 0.0043,0.0285 ± 0.0006
8,Congestive heart failure; nonhypertensive,0.7588 ± 0.0027,0.262 ± 0.0069
9,Coronary atherosclerosis and other heart disease,0.7265 ± 0.0054,0.0707 ± 0.0043


## 5. iTransformer

In [12]:
import sys
sys.path.insert(0, "/home/DAHS2/DAHS_EHR/iTransformer")

from layers.Transformer_EncDec import Encoder, EncoderLayer
from layers.SelfAttention_Family import FullAttention, AttentionLayer

class MultiTaskITransformer(nn.Module):
    def __init__(self, input_size=None, seq_len=WINDOW_HOURS, hidden_size=128, n_heads=8, e_layers=2, d_ff=256, dropout=0.2, n_binary=9, n_sofa=6, n_sofa_classes=4, n_pheno=25):
        super().__init__()
        self.seq_len = seq_len
        self.value_embedding = nn.Linear(seq_len * 2, hidden_size)
        self.embedding_dropout = nn.Dropout(dropout)
        self.encoder = Encoder(
            [
                EncoderLayer(
                    AttentionLayer(
                        FullAttention(False, factor=1, attention_dropout=dropout, output_attention=False),
                        hidden_size, n_heads,
                    ),
                    hidden_size, d_ff, dropout=dropout, activation="gelu",
                )
                for _ in range(e_layers)
            ],
            norm_layer=nn.LayerNorm(hidden_size),
        )
        self.pool_proj = nn.Linear(hidden_size, hidden_size)
        self.pool_query = nn.Parameter(torch.randn(hidden_size) / hidden_size ** 0.5)

        self.dropout = nn.Dropout(dropout)
        self.binary_head = nn.Linear(hidden_size, n_binary)
        self.sofa_head = nn.Linear(hidden_size, n_sofa * n_sofa_classes)
        self.pheno_head = nn.Linear(hidden_size, n_pheno)
        self.n_sofa = n_sofa
        self.n_sofa_classes = n_sofa_classes

    def forward(self, x):
        C = x.shape[-1] // 2
        val, mask = x[..., :C], x[..., C:]

        obs_count = mask.sum(dim=1, keepdim=True).clamp(min=1)
        means = (val * mask).sum(dim=1, keepdim=True) / obs_count
        val_centered = (val - means) * mask
        var = (val_centered ** 2).sum(dim=1, keepdim=True) / obs_count
        stdev = torch.sqrt(var + 1e-5)
        val_norm = (val_centered / stdev) * mask

        tokens_in = torch.cat([val_norm.permute(0, 2, 1), mask.permute(0, 2, 1)], dim=-1)

        h = self.embedding_dropout(self.value_embedding(tokens_in))
        h, _ = self.encoder(h, attn_mask=None)

        # attention pooling over the C variate tokens
        scores = self.pool_proj(h) @ self.pool_query / (h.shape[-1] ** 0.5)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        pooled = (h * weights).sum(dim=1)

        pooled = self.dropout(pooled)
        binary_logits = self.binary_head(pooled)
        sofa_logits = self.sofa_head(pooled).view(-1, self.n_sofa, self.n_sofa_classes)
        pheno_logits = self.pheno_head(pooled)
        return binary_logits, sofa_logits, pheno_logits


In [13]:
itransformer_binary_results, itransformer_sofa_results, itransformer_phenotype_results = [], [], []
itransformer_phenotype_raw = {}

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MultiTaskITransformer(input_size=n_features)
    model, best_val_score = train_multitask(model, mt_train_loader, mt_val_loader, device)
    binary_rows, sofa_rows, pheno_rows, pheno_true, pheno_probs = evaluate_multitask(model, mt_test_loader, device)

    for row in binary_rows:
        itransformer_binary_results.append({**row, "seed": seed})
    for row in sofa_rows:
        itransformer_sofa_results.append({**row, "seed": seed})
    for i, label in enumerate(ALL_PHENO):
        itransformer_phenotype_results.append({**pheno_rows[i], "seed": seed})
        itransformer_phenotype_raw[(label, seed)] = (pheno_true[:, i], pheno_probs[:, i])

itransformer_binary_results = pd.DataFrame(itransformer_binary_results)
itransformer_sofa_results = pd.DataFrame(itransformer_sofa_results)
itransformer_phenotype_results = pd.DataFrame(itransformer_phenotype_results)

summarize_seeds(itransformer_binary_results, ["auroc", "auprc"])

epoch:   0%|          | 0/60 [00:00<?, ?it/s]

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

...

train:   0%|          | 0/1276 [00:00<?, ?it/s]

val:   0%|          | 0/80 [00:00<?, ?it/s]

early stopping


evaluate:   0%|          | 0/100 [00:00<?, ?it/s]

,label,auroc,auprc
0,los_3days,0.6814 ± 0.0005,0.5689 ± 0.0011
1,los_7days,0.7069 ± 0.0025,0.2367 ± 0.0031
2,mortality_48hr,0.7456 ± 0.0011,0.1022 ± 0.0073
3,mortality_inicu,0.7426 ± 0.0021,0.1763 ± 0.0088
4,readmission_30,0.5775 ± 0.0042,0.1082 ± 0.001
5,shock_8hr,0.8536 ± 0.0024,0.0985 ± 0.0032
6,transfusion_12hr,0.655 ± 0.0068,0.0341 ± 0.0034
7,vasopressor_need_12hr,0.7925 ± 0.0025,0.2525 ± 0.0059
8,ventilator_need_12hr,0.9353 ± 0.0088,0.0169 ± 0.0025


In [14]:
summarize_seeds(itransformer_sofa_results, ["auroc_macro", "auprc_macro"])

,label,auroc_macro,auprc_macro
0,SOFA_cardiovascular_24hr,0.7835 ± 0.0041,0.3097 ± 0.0018
1,SOFA_centralnervous_24hr,0.6845 ± 0.0007,0.3479 ± 0.0009
2,SOFA_coagulation_24hr,0.7016 ± 0.0029,0.3243 ± 0.0024
3,SOFA_liver_24hr,0.7345 ± 0.0009,0.2813 ± 0.0012
4,SOFA_renal_24hr,0.6575 ± 0.0035,0.3083 ± 0.0019
5,SOFA_respiratory_24hr,0.785 ± 0.0011,0.4409 ± 0.0011


In [15]:
itransformer_per_label_summary = summarize_seeds(itransformer_phenotype_results, ["auroc", "auprc"])

macro_per_seed = itransformer_phenotype_results.groupby("seed")[["auroc", "auprc"]].mean()
macro_mean, macro_std = macro_per_seed.mean(), macro_per_seed.std()

# included_labels = itransformer_phenotype_results["label"].unique().tolist()
# micro_rows = []
# for seed in SEEDS:
#     y_all = np.concatenate([lstm_phenotype_raw[(label, seed)][0] for label in included_labels])
#     p_all = np.concatenate([lstm_phenotype_raw[(label, seed)][1] for label in included_labels])
#     micro_rows.append({
#         "seed": seed,
#         "auroc": roc_auc_score(y_all, p_all),
#         "auprc": average_precision_score(y_all, p_all),
#     })
# micro_per_seed = pd.DataFrame(micro_rows).set_index("seed")
# micro_mean, micro_std = micro_per_seed.mean(), micro_per_seed.std()

overall_rows = pd.DataFrame([
    {
        "label": "OVERALL (macro avg)",
        "auroc": f"{macro_mean['auroc']:.4f} ± {macro_std['auroc']:.4f}",
        "auprc": f"{macro_mean['auprc']:.4f} ± {macro_std['auprc']:.4f}",
    },
    # {
    #     "label": "OVERALL (micro avg)",
    #     "auroc": f"{micro_mean['auroc']:.4f} ± {micro_std['auroc']:.4f}",
    #     "auprc": f"{micro_mean['auprc']:.4f} ± {micro_std['auprc']:.4f}",
    # },
])

itransformer_phenotype_summary = pd.concat([itransformer_per_label_summary, overall_rows], ignore_index=True)
itransformer_phenotype_summary

,label,auroc,auprc
0,Acute and unspecified renal failure,0.6573 ± 0.0025,0.2105 ± 0.0022
1,Acute cerebrovascular disease,0.6275 ± 0.0003,0.1166 ± 0.0023
2,Acute myocardial infarction,0.555 ± 0.0084,0.0649 ± 0.0026
3,Cardiac dysrhythmias,0.5401 ± 0.0053,0.1498 ± 0.0033
4,Chronic kidney disease,0.5735 ± 0.0062,0.1194 ± 0.0022
5,Chronic obstructive pulmonary disease and bron...,0.5737 ± 0.0038,0.1028 ± 0.0024
6,Complications of surgical procedures or medica...,0.5965 ± 0.0011,0.013 ± 0.0006
7,Conduction disorders,0.5372 ± 0.02,0.01 ± 0.0005
8,Congestive heart failure; nonhypertensive,0.5682 ± 0.0056,0.1246 ± 0.0024
9,Coronary atherosclerosis and other heart disease,0.6114 ± 0.004,0.0436 ± 0.0006


## 6. STraTS

In [25]:
df_irregular_train = pd.read_pickle("/home/DAHS2/DAHS_EHR/datasets/new_data_preparation/finetune_train_24_fold0_eicu.pkl")
df_irregular_val = pd.read_pickle("/home/DAHS2/DAHS_EHR/datasets/new_data_preparation/finetune_val_24_fold0_eicu.pkl")
df_irregular_test = pd.read_pickle("/home/DAHS2/DAHS_EHR/datasets/new_data_preparation/finetune_test_24_eicu.pkl")